## **feature extraction**

In [2]:
import pandas as pd

In [3]:
data = pd.read_csv('data/wine_cleared.csv', sep=',', index_col=False)
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 129971 entries, 0 to 129970
Data columns (total 13 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Unnamed: 0             129971 non-null  int64  
 1   country                129908 non-null  str    
 2   description            129971 non-null  str    
 3   designation            129971 non-null  str    
 4   points                 129971 non-null  int64  
 5   price                  129971 non-null  float64
 6   province               129908 non-null  str    
 7   region_1               129971 non-null  str    
 8   taster_name            129971 non-null  str    
 9   taster_twitter_handle  129971 non-null  str    
 10  title                  129971 non-null  str    
 11  variety                129970 non-null  str    
 12  winery                 129971 non-null  str    
dtypes: float64(1), int64(2), str(10)
memory usage: 12.9 MB


In [4]:
data.drop(labels='Unnamed: 0', axis=1, inplace=True)

### **numeric features**

In [5]:
# для удобства сразу преобразуем признак в int
data['price_round'] = data['price'].round().astype(int)

### **text features**

In [6]:
regex = '(\\d{4})'
data['year'] = data['title'].str.extract(pat=regex).astype(float)

### **categoty features**

In [7]:
data['is_usa'] = data['country'].apply(lambda x: 1 if x == 'US' else 0)

### *a bit of practice*

In [8]:
# find the most popular country after USA
display(data['country'].value_counts(ascending=False).iloc[:5])
# extract feature is_france and is_italy 
data['is_france'] = data['country'].apply(lambda x: 1 if x == 'France' else 0)
data['is_italy'] = data['country'].apply(lambda x: 1 if x == 'Italy' else 0)
# how many wine made in France and Italy
print('France', data['is_france'].sum())
print('Italy', data['is_italy'].sum())
# age of wine if wine elder then 2010 - 1 else 0
data['old_wine'] = data['year'].apply(lambda x: 1 if x < 2010 else 0)
print('old wine', data['old_wine'].sum())
# extract locality
regex_0 = '\\((.+)\\)'
data['locality'] = data['title'].str.extract(pat=regex_0)

country
US          54504
France      22093
Italy       19540
Spain        6645
Portugal     5691
Name: count, dtype: int64

France 22093
Italy 19540
old wine 39781


### **additional data sources** 

In [9]:
country_population = pd.read_csv('data/country_population.csv', sep=';', index_col=False)
country_population.info()
data = data.join(country_population.set_index('country'), on='country')

<class 'pandas.DataFrame'>
RangeIndex: 241 entries, 0 to 240
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   country     241 non-null    str  
 1   population  241 non-null    str  
dtypes: str(2)
memory usage: 3.9 KB


In [10]:
country_area = pd.read_csv('data/country_area.csv', sep=';', index_col=False)
data = data.join(country_area.set_index('country'), on='country')

### **feature time difference between wine production and 12.01.2022**

In [19]:
data['year'] = pd.to_datetime(data['year'], errors = 'coerce')
data['years_diff'] = (pd.to_datetime("2022-01-12") - data['year']).dt.days
data['years_diff'].max()

np.float64(19003.0)